# 06 · Visualización interactiva y tableros — Adopción de IA en empresas

Este cuaderno continúa el trabajo sobre el panel de empresas (`ai_company_adoption_limpio.parquet`). Construye un tablero con Plotly para explorar `ai_adoption_rate` según industria, país y escala, agregando los datos previamente para no sobrecargar el navegador y mostrando la dispersión e incertidumbre de las estimaciones.

## Pregunta, unidad y alcance

* **Pregunta:** ¿Cómo varía la adopción de IA entre industrias y países a lo largo del tiempo, y qué relación guarda con la escala de la empresa?
* **Unidad de análisis:** una fila por empresa y trimestre (panel 2023–2026).
* **Objetivo:** tablero de control para comparar sectores y regiones sobre una misma escala, sin truncar ejes ni ocultar el tamaño de muestra de cada grupo.
* **Datos y transformaciones:** se trabaja sobre agregaciones para evitar graficar 150.000 puntos en el cliente. La relación entre personal y adopción usa escala logarítmica para manejar la asimetría de la cola.

In [2]:
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# 1. Búsqueda automática del archivo en el directorio actual o superiores
nombres_posibles = [
    'ai_company_adoption_limpio.parquet',
    'ai_company_adoption.csv'
]

ruta_encontrada = None
directorios_a_buscar = [Path.cwd(), *Path.cwd().parents]

for directorio in directorios_a_buscar:
    for nombre in nombres_posibles:
        # Buscar coincidencia exacta en subcarpetas
        coincidencias = list(directorio.glob(f"**/{nombre}"))
        if coincidencias:
            ruta_encontrada = coincidencias[0]
            break
    if ruta_encontrada:
        break

if ruta_encontrada is None:
    raise FileNotFoundError(
        "No se encontró 'ai_company_adoption_limpio.parquet' ni 'ai_company_adoption.csv'. "
        f"Directorio actual de ejecución: {Path.cwd()}"
    )

print(f"Archivo localizado en: {ruta_encontrada}")

# 2. Carga según extensión
if ruta_encontrada.suffix == '.parquet':
    df = pd.read_parquet(ruta_encontrada)
    print("Dataset cargado desde archivo Parquet.")
else:
    df = pd.read_csv(ruta_encontrada)
    print("Dataset cargado desde archivo CSV.")

# 3. Normalización de columnas y tipos para el proyecto
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

df['survey_year'] = pd.to_numeric(df['survey_year'], errors='coerce').astype(int)
df['ai_adoption_rate'] = pd.to_numeric(df['ai_adoption_rate'], errors='coerce')
df['num_employees'] = pd.to_numeric(df['num_employees'], errors='coerce')
df['annual_revenue_usd_millions'] = pd.to_numeric(df['annual_revenue_usd_millions'], errors='coerce')

print(f"Total registros: {len(df):,} | Empresas únicas: {df['company_id'].nunique():,} | Años: {df['survey_year'].min()}-{df['survey_year'].max()}")

Archivo localizado en: c:\Users\Usuario\Downloads\Lab06\aa-2026-entorno\proyecto_equipo\datos\derivados\ai_company_adoption_limpio.parquet
Dataset cargado desde archivo Parquet.
Total registros: 150,000 | Empresas únicas: 10,000 | Años: 2023-2026


In [3]:
# Selección de las 5 industrias y 6 países más representados
industrias_principales = df.groupby('industry')['company_id'].count().nlargest(5).index.tolist()
paises_principales = df.groupby('country')['company_id'].count().nlargest(6).index.tolist()

df_tablero = df[df['industry'].isin(industrias_principales) & df['country'].isin(paises_principales)].copy()

print("Industrias seleccionadas:", industrias_principales)
print("Países seleccionados:", paises_principales)
print(f"Registros en el tablero: {len(df_tablero):,} ({len(df_tablero)/len(df):.1%} del total)")

Industrias seleccionadas: ['Technology', 'Finance', 'Healthcare', 'Manufacturing', 'Retail']
Países seleccionados: ['France', 'Japan', 'South Korea', 'Germany', 'Chile', 'Egypt']
Registros en el tablero: 20,032 (13.4% del total)


## 1. Evolución del indicador por sector

Se calcula la tasa media de adopción por año e industria. En el detalle bajo demanda (`hover_data`) se incorpora la cantidad de empresas observadas en cada punto para dar contexto a las variaciones entre períodos.

In [4]:
# Agrupación y reset_index explícito para evitar problemas de alineación
evolucion = (
    df_tablero.groupby(['survey_year', 'industry'])
    .agg({
        'ai_adoption_rate': 'mean',
        'company_id': 'nunique'
    })
    .rename(columns={
        'ai_adoption_rate': 'adopcion_media',
        'company_id': 'total_empresas'
    })
    .reset_index()
)

fig1 = px.line(
    evolucion,
    x='survey_year',
    y='adopcion_media',
    color='industry',
    markers=True,
    hover_data={'total_empresas': True, 'adopcion_media': ':.2f'},
    labels={
        'survey_year': 'Año',
        'adopcion_media': 'Tasa media de adopción (%)',
        'industry': 'Industria',
        'total_empresas': 'Empresas observadas'
    },
    title='Evolución de la adopción media de IA por industria'
)

fig1.update_yaxes(range=[0, 100])
fig1.update_layout(hovermode='x unified', height=400)
fig1.show()

In [5]:
# Comparación del costo en memoria entre datos crudos y agregados
muestra = df_tablero.sample(min(4000, len(df_tablero)), random_state=1)

crudo = px.scatter(muestra, x='num_employees', y='ai_adoption_rate')
agregado = px.line(evolucion, x='survey_year', y='adopcion_media', color='industry')

kb_crudo = len(crudo.to_json()) / 1024
kb_agreg = len(agregado.to_json()) / 1024

print(f'Dispersión con {len(muestra):,} puntos : {kb_crudo:8.1f} KB')
print(f'Líneas agregadas ({len(evolucion):,} puntos): {kb_agreg:8.1f} KB')
print(f'Relación: {kb_crudo / kb_agreg:.0f} a 1')

bytes_por_punto = kb_crudo * 1024 / len(muestra)
mb_sin_agregar = 3 * bytes_por_punto * len(df) / (1024 ** 2)
print(f'Costo estimado de 3 vistas sin agregar sobre las {len(df):,} filas: {mb_sin_agregar:.1f} MB')

Dispersión con 4,000 puntos :     60.9 KB
Líneas agregadas (36 puntos):     10.7 KB
Relación: 6 a 1
Costo estimado de 3 vistas sin agregar sobre las 150,000 filas: 6.7 MB


In [9]:
# 1. Asegurar que 'country' sea string y no mantenga categorías vacías
datos_pais = (
    df_tablero[df_tablero['country'].isin(paises_principales)]
    .copy()
)
datos_pais['country'] = datos_pais['country'].astype(str)

# Agrupar por año y país
evolucion_pais = (
    datos_pais.groupby(['survey_year', 'country'])['ai_adoption_rate']
    .mean()
    .reset_index()
)

# 2. Crear una grilla limpia de 2 filas x 3 columnas (para los 6 países)
fig2 = make_subplots(
    rows=2, cols=3,
    subplot_titles=paises_principales,
    shared_yaxes=True,
    vertical_spacing=0.18,
    horizontal_spacing=0.08
)

for i, pais in enumerate(paises_principales):
    fila = (i // 3) + 1
    col = (i % 3) + 1
    
    sub = evolucion_pais[evolucion_pais['country'] == pais]
    
    fig2.add_trace(
        go.Scatter(
            x=sub['survey_year'],
            y=sub['ai_adoption_rate'],
            mode='lines+markers',
            name=pais,
            line=dict(color='#1F3864', width=2),
            marker=dict(size=6),
            showlegend=False
        ),
        row=fila, col=col
    )
    
    # Eje X con los años del relevamiento
    fig2.update_xaxes(
        tickvals=[2023, 2024, 2025, 2026],
        title_text='Año' if fila == 2 else '',
        row=fila, col=col
    )
    
    # Eje Y de 0 a 100% fijo (solo ponemos título en la primera columna para no encimar)
    fig2.update_yaxes(
        range=[0, 100],
        title_text='Adopción (%)' if col == 1 else '',
        row=fila, col=col
    )

fig2.update_layout(
    height=520,
    title_text='Evolución de adopción de IA por país (ejes compartidos)',
    margin=dict(l=60, r=30, t=80, b=50)
)

fig2.show()

In [10]:
fig3 = go.Figure()

for ind in industrias_principales:
    datos_ind = evolucion[evolucion['industry'] == ind]
    fig3.add_trace(go.Scatter(
        x=datos_ind['survey_year'],
        y=datos_ind['adopcion_media'],
        name=ind,
        mode='lines+markers',
        line=dict(width=2),
        marker=dict(size=6),
        visible=(ind == industrias_principales[0])
    ))

botones = []
for i, ind in enumerate(industrias_principales):
    visibles = [j == i for j in range(len(industrias_principales))]
    botones.append(dict(
        label=ind,
        method='update',
        args=[{'visible': visibles}, {'title': f'Adopción de IA — Sector: {ind}'}]
    ))

botones.append(dict(
    label='Todas las industrias',
    method='update',
    args=[{'visible': [True] * len(industrias_principales)}, {'title': 'Adopción de IA — Todos los sectores'}]
))

fig3.update_layout(
    title=f'Adopción de IA — Sector: {industrias_principales[0]}',
    xaxis_title='Año',
    yaxis_title='Tasa media de adopción (%)',
    yaxis_range=[0, 100],
    updatemenus=[dict(buttons=botones, direction='down', x=1.0, y=1.2, xanchor='right')],
    height=420
)
fig3.show()

## 2. Tablero integrado en una pantalla

Estructura de tres niveles sin desplazamiento vertical:
1. **Fila 1 (Tarjetas):** adopción media, total de empresas e ingresos medios del último año junto con su variación interanual.
2. **Fila 2 (Tendencia):** evolución temporal por industria con ejes compartidos.
3. **Fila 3 (Cortes):** ranking por país con eje desde cero a la izquierda, y dispersión respecto al total de empleados a la derecha.

In [ ]:
ultimo_anio = int(df_tablero['survey_year'].max())
actual = df_tablero[df_tablero['survey_year'] == ultimo_anio]
anterior = df_tablero[df_tablero['survey_year'] == ultimo_anio - 1]

# Asegurar que 'country' sea string para evitar problemas de categorías vacías
df_tablero['country'] = df_tablero['country'].astype(str)

kpis = {
    'Adopción Promedio (%)': (actual['ai_adoption_rate'].mean(), anterior['ai_adoption_rate'].mean()),
    'Empresas en Panel': (actual['company_id'].nunique(), anterior['company_id'].nunique()),
    'Ingresos Medios (M USD)': (actual['annual_revenue_usd_millions'].mean(), anterior['annual_revenue_usd_millions'].mean())
}

# Ranking por país
ranking_pais = (
    df_tablero.groupby('country')['ai_adoption_rate']
    .mean()
    .loc[paises_principales]
    .sort_values()
)

# Muestra para dispersión (evita sobrecarga del navegador)
muestra_disp = df_tablero.sample(n=min(1500, len(df_tablero)), random_state=1)

tablero = make_subplots(
    rows=3, cols=3,
    specs=[
        [{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}],
        [{'type': 'xy', 'colspan': 3}, None, None],
        [{'type': 'xy', 'colspan': 2}, None, {'type': 'xy'}]
    ],
    row_heights=[0.16, 0.44, 0.40],
    vertical_spacing=0.10,
    horizontal_spacing=0.08,
    subplot_titles=[
        '', '', '',
        'Adopción media por industria a lo largo del tiempo (%)',
        'Ranking de adopción media por país',
        'Empleados vs. Tasa de Adopción'
    ]
)

# 1. Fila de KPIs
for i, (nombre, (val_act, val_ant)) in enumerate(kpis.items(), start=1):
    tablero.add_trace(go.Indicator(
        mode='number+delta',
        value=val_act,
        title={'text': nombre, 'font': {'size': 12}},
        number={'valueformat': ',.1f', 'font': {'size': 30}},
        delta={'reference': val_ant, 'relative': True, 'valueformat': '.1%'}
    ), row=1, col=i)

# 2. Fila de evolución
for ind in industrias_principales:
    datos_ind = evolucion[evolucion['industry'] == ind]
    tablero.add_trace(go.Scatter(
        x=datos_ind['survey_year'],
        y=datos_ind['adopcion_media'],
        name=ind,
        mode='lines+markers',
        legendgroup='industria'
    ), row=2, col=1)

# 3. Fila inferior: Ranking horizontal por país
tablero.add_trace(go.Bar(
    x=ranking_pais.values,
    y=ranking_pais.index.tolist(),
    orientation='h',
    marker_color='#1F3864',
    showlegend=False
), row=3, col=1)

# 4. Fila inferior: Dispersión escala vs adopción
tablero.add_trace(go.Scatter(
    x=muestra_disp['num_employees'],
    y=muestra_disp['ai_adoption_rate'],
    mode='markers',
    marker=dict(size=4, color=muestra_disp['ai_adoption_rate'], colorscale='Blues'),
    showlegend=False,
    hovertemplate='Empleados: %{x:,}<br>Adopción: %{y:.1f}%<extra></extra>'
), row=3, col=3)

# Ajustes de ejes
tablero.update_xaxes(title_text='Año', row=2, col=1)
tablero.update_yaxes(title_text='Adopción (%)', range=[0, 100], row=2, col=1)
tablero.update_xaxes(title_text='Adopción (%)', range=[0, 100], row=3, col=1)
tablero.update_xaxes(title_text='Empleados (escala log)', type='log', row=3, col=3)
tablero.update_yaxes(title_text='Adopción (%)', range=[0, 100], row=3, col=3)

tablero.update_layout(
    height=850,
    title_text=f'Tablero de Control: Adopción de IA (Período {ultimo_anio})',
    margin=dict(t=90, b=40, l=60, r=60)
)
tablero.show()

## 3. Dispersión y tamaño muestral

Los promedios se muestran junto con su intervalo de confianza al 95% y el número de observaciones ($n$). Si los intervalos de dos sectores se solapan, la diferencia de orden en el ranking no es concluyente con los datos observados.

In [12]:
stats_industria = (
    df_tablero.groupby('industry')['ai_adoption_rate']
    .agg(media='mean', desvio='std', n='count')
    .sort_values('media')
)

stats_industria['error_estandar'] = stats_industria['desvio'] / np.sqrt(stats_industria['n'])
stats_industria['ic95'] = 1.96 * stats_industria['error_estandar']

display(stats_industria[['media', 'desvio', 'n', 'ic95']].round(2))

fig4 = px.bar(
    stats_industria.reset_index(),
    x='media',
    y='industry',
    orientation='h',
    error_x='ic95',
    text='n',
    labels={'media': 'Adopción Media (%)', 'industry': '', 'n': 'Casos (n)'},
    title='Adopción media por industria con intervalo de confianza del 95%'
)

fig4.update_traces(marker_color='#1F3864', texttemplate='n=%{text:,}', textposition='outside')
fig4.update_xaxes(range=[0, 100])
fig4.update_layout(height=380, margin=dict(r=90))
fig4.show()

,media,desvio,n,ic95
industry,,,,
Retail,33.69,13.93,3758,0.45
Manufacturing,34.51,13.63,3919,0.43
Healthcare,34.64,13.77,3736,0.44
Finance,36.98,14.36,4620,0.41
Technology,41.89,13.90,3999,0.43
Agriculture,NaN,NaN,0,NaN
Consulting,NaN,NaN,0,NaN
Education,NaN,NaN,0,NaN
Logistics,NaN,NaN,0,NaN
